# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Select a Colab T4 GPU before running the model sections.


In [ ]:
import gc
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/vbwork2/K4-Track-4-Day10-VisionLanguageModel.git"
CODE_REF = "feat/day10-kaggle-qlora"
REPO = Path.cwd()
if not (REPO / "scripts").is_dir():
    REPO = Path("/content") / "K4-Track-4-Day10-VisionLanguageModel"
    if not REPO.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", CODE_REF, REPO_URL, str(REPO)], check=True)
    else:
        branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO, text=True).strip()
        if branch != CODE_REF:
            raise RuntimeError(f"Expected branch {CODE_REF}, found {branch}")
os.chdir(REPO)
sys.path.insert(0, str(REPO))

try:
    from google.colab import userdata
    for secret_name in ("KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN", "HF_REPO_ID"):
        try:
            os.environ[secret_name] = userdata.get(secret_name)
        except Exception:
            pass
except ImportError:
    pass

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "huggingface_hub", "kagglehub"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "student_tests", "--basetemp", "/content/day10_pytest_tmp"], cwd=REPO, check=True)
DATA_DIR = REPO / "data"
RUN_ROOT = REPO / "work" / "day10_runs"
REMOTE_REPO_ID = os.environ.get("HF_REPO_ID")
HF_TOKEN = os.environ.get("HF_TOKEN")
print("Repository:", REPO)
print("CPU checks passed. HF backup is configured:", bool(REMOTE_REPO_ID and HF_TOKEN))


## 1. Dữ liệu: tải và chia split

In [ ]:
from student.data import prepare_public_dataset

DATASET = prepare_public_dataset(REPO, DATA_DIR, REPO / "work" / "day10_prep")


In [ ]:
from student.data import validate_splits
from student.visualization import render_ground_truth_overlays

DATASET = validate_splits(DATA_DIR, check_expected=True)
print("dataset_version:", DATASET["meta"]["dataset_version"])
print("split summary:", DATASET["summary"])
print("sessions:", len(DATASET["sessions"]))
LABEL_OVERLAYS = render_ground_truth_overlays(
    DATASET["rows"]["train"], DATA_DIR, REPO / "work" / "day10_overlays", limit=6
)
print("Six label overlays:", LABEL_OVERLAYS)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

from student.labels import grounding_box_to_pixels, objects_to_answer, yolo_to_box

sample = DATASET["rows"]["train"][0]
answer = objects_to_answer(sample["objects"], sample["width"], sample["height"])
boxes = json.loads(answer)
print("image_id:", sample["image_id"])
print("answer:", answer)
if boxes:
    print("first box in pixels:", grounding_box_to_pixels(boxes[0], sample["width"], sample["height"]))


## 3. E0: zero-shot

In [ ]:
import torch

from student.experiments import run_experiment
from student.inference import run_incremental_inference
from student.model import load_model_bundle

assert torch.cuda.is_available(), "Select a T4 GPU runtime before E0 inference."
e0_bundle = load_model_bundle("E0", trainable=False)
PROCESSOR = e0_bundle.processor
E0_PREDICTIONS = REPO / "work" / "e0.jsonl"
run_incremental_inference(
    DATASET["rows"]["test"], DATA_DIR, PROCESSOR, e0_bundle.model, E0_PREDICTIONS,
    device="cuda:0", max_new_tokens=256,
)
del e0_bundle
gc.collect()
torch.cuda.empty_cache()
print("E0 predictions:", E0_PREDICTIONS)


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from PIL import Image
from transformers import AutoProcessor

from student.dataset import assert_image_token_count, collate_examples, encode_example
from student.labels import objects_to_answer

if "PROCESSOR" not in globals():
    PROCESSOR = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
example = DATASET["rows"]["train"][0]
with Image.open(DATA_DIR / example["image_path"]) as source:
    image = source.convert("RGB")
answer = objects_to_answer(example["objects"], example["width"], example["height"])
encoded = encode_example(PROCESSOR, image, answer)
print("image token count:", assert_image_token_count(PROCESSOR, encoded, expected=220))
print("answer tokens:", PROCESSOR.tokenizer.decode(encoded["labels"][encoded["labels"] != -100], skip_special_tokens=False))
print("batch fields:", {key: tuple(value.shape) for key, value in collate_examples([encoded], PROCESSOR).items()})


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from student.experiments import load_experiment_config
from student.model import EXPECTED_ARCHITECTURE_COUNTS

for setup in ("E1", "E2", "E3"):
    config = load_experiment_config(REPO, setup)
    print(setup, "expected LLM/ViT modules:", EXPECTED_ARCHITECTURE_COUNTS,
          "projector trainable:", config.get("train_projector"))
print("The shared model loader asserts actual module names before attaching LoRA.")


## 6. Vòng huấn luyện

In [ ]:
import gc
import torch

from student.experiments import run_experiment

assert torch.cuda.is_available(), "Select a T4 GPU runtime before supervised training."
TRAIN_RESULTS = {}
for setup in ("E1", "E2", "E3"):
    result = run_experiment(
        setup, REPO, DATA_DIR, RUN_ROOT, split="val",
        micro_batch_size=1, grad_accum=8, checkpoint_every=25,
        remote_repo_id=REMOTE_REPO_ID, token=HF_TOKEN, remote_required=True,
        evaluate=False,
    )
    TRAIN_RESULTS[setup] = {"run_id": result["run_id"], "training": result["training"]}
    print(setup, TRAIN_RESULTS[setup])
    del result
    gc.collect()
    torch.cuda.empty_cache()


## 7. Suy luận và chấm E1–E3

In [ ]:
import gc
import torch

from student.experiments import run_experiment

EVALUATION_RESULTS = {}
for setup in ("E1", "E2", "E3"):
    result = run_experiment(
        setup, REPO, DATA_DIR, RUN_ROOT, split="test",
        micro_batch_size=1, grad_accum=8, checkpoint_every=25,
        remote_repo_id=REMOTE_REPO_ID, token=HF_TOKEN, remote_required=True,
        evaluate=True,
    )
    EVALUATION_RESULTS[setup] = result["evidence"]
    print(setup, result["evidence"])
    print("raw examples:", result["evidence"]["raw_examples"])
    del result
    gc.collect()
    torch.cuda.empty_cache()


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
import json

from student.experiments import compare_rank_evidence, run_rank_ablation
from student.reward import iou_reward

E4_RESULTS = run_rank_ablation(
    REPO, DATA_DIR, RUN_ROOT, base_setup="E1", split="test",
    micro_batch_size=1, grad_accum=8, checkpoint_every=25,
    remote_repo_id=REMOTE_REPO_ID, token=HF_TOKEN, remote_required=True,
)
E4_COMPARISON = compare_rank_evidence(
    REPO / "report" / "evidence" / "e4_rank_4_base_e1.json",
    REPO / "report" / "evidence" / "e4_rank_64_base_e1.json",
)
print(json.dumps(E4_COMPARISON, ensure_ascii=False, indent=2))
assert iou_reward("[]", []) == 1.0
print("E5 reward CPU check passed. GRPO is optional and was not run in this notebook cell.")
